# KRITIS–Philippines: tropical cyclone × grid-bus screening (canonical)

This notebook is the **canonical** version of the Philippines tropical-cyclone × grid-bus screening workflow.

It keeps only the parts that are methodologically central and reusable:

1. **Canonical baseline index**  
   - IBTrACS West Pacific storm tracks
   - Philippines screening box
   - WorldPop-style gridded population
   - PyPSA-PH buses
   - bus weights from voltage + local population + light network adjustment
   - event score from **max-over-time bus hazard** summed over buses

2. **Geographic refinement**  
   - on-land tagging and land-vs-sea split

3. **Robustness**  
   - parameter grid over search radius and decay
   - top-10 overlap and Spearman rank similarity

4. **External validation interface**  
   - benchmark table export
   - optional joined validation against a curated external master file

5. **Functional extension (kept as the only canonical extension)**  
   - hazard → failure probability
   - expected population / infrastructure / network consequence
   - soft fragmentation proxy
   - master comparison table across baseline / population / network / functional views

## What was intentionally removed

The following were **not** kept in the canonical notebook because they are either duplicative, presentation-only, or exploratory branches that should live elsewhere:

- duplicate population-merge cells
- slide animation generation
- presentation-specific plotting block
- multiple alternative hazard proxies (vortex / coastal / rain)
- overlapping cascade proxy variants
- duplicate network fragmentation formulations
- raw EM-DAT alias-building ETL workflow
- ad hoc one-off exploratory plots that do not change the main interpretation

## Interpretation discipline

- `impact_score` is a **screening index**, not a damage or outage model.
- Functional and fragmentation outputs are **consequence proxies**, not operator-grade contingency analysis.
- All rankings should be interpreted as **relative event ordering within the screened catalogue**.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from tropycal import tracks
from scipy.stats import spearmanr
from itertools import product

REPO_ROOT = Path.cwd().resolve()
XYZ_PATH = REPO_ROOT / "data/worldpop_ph_2020_1km/phl_pd_2020_1km_ASCII_XYZ.csv"
BUSES_CSV = REPO_ROOT / "PyPSA-PH/data/buses.csv"
LINES_CSV = REPO_ROOT / "PyPSA-PH/data/lines.csv"
GENERATORS_CSV = REPO_ROOT / "PyPSA-PH/data/generators.csv"
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2000

PH_BOUNDS = {
    "lat_min": 4.0,
    "lat_max": 22.0,
    "lon_min": 116.0,
    "lon_max": 128.0,
}

SEARCH_RADIUS_KM = 150.0
DECAY_KM = 30.0
ALPHA_VNOM = 0.5
BUS_POP_RADIUS_KM = 50.0

# light network adjustment for canonical baseline bus weight
ALPHA_NET_DEG = 0.15
ALPHA_NET_GEN = 0.10

RANDOM_SEED = 42
POP_PLOT_SAMPLE_N = 150_000
PLACEBO_N = 30

# functional extension parameters
FAIL_CENTER = 0.35
FAIL_STEEPNESS = 12.0

print("Configuration loaded.")
print("REPO_ROOT:", REPO_ROOT)

## Helpers

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    earth_radius_km = 6371.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2.0 * np.arcsin(np.sqrt(a))
    return earth_radius_km * c


def storm_name_map_from_ib(ib_ds):
    m = {}
    for sid, storm in ib_ds.data.items():
        nm = storm.get("name")
        if nm is None or str(nm).strip() == "":
            nm = "UNNAMED"
        m[str(sid)] = str(nm).upper()
    return m


def load_worldpop_xyz(path, verbose=True):
    try:
        df = pd.read_csv(path)
        if df.shape[1] < 3:
            raise ValueError("Too few columns from default CSV read.")
    except (pd.errors.ParserError, ValueError):
        df = pd.read_csv(
            path,
            sep=r"\s+",
            header=None,
            names=["c1", "c2", "c3"],
            comment="#",
            engine="python",
        )

    df.columns = [str(c).strip() for c in df.columns]
    if verbose:
        print("Raw columns:", list(df.columns))
        display(df.head())

    colmap = {str(c).strip().lower(): c for c in df.columns}

    def pick(candidates):
        for c in candidates:
            if c in colmap:
                return colmap[c]
        return None

    lon_col = pick(["lon", "longitude", "x"])
    lat_col = pick(["lat", "latitude", "y"])
    pop_col = pick(["pop_density", "population", "pop", "z", "value", "density"])

    if lon_col is None or lat_col is None or pop_col is None:
        print("Falling back to first three columns.")
        first3 = list(df.columns)[:3]
        if len(first3) < 3:
            raise ValueError("Input file does not contain at least three columns.")
        lon_col, lat_col, pop_col = first3[0], first3[1], first3[2]

    out = df[[lon_col, lat_col, pop_col]].copy()
    out.columns = ["lon", "lat", "pop_value"]
    for c in ["lon", "lat", "pop_value"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out = out.dropna(subset=["lon", "lat", "pop_value"]).copy()
    if (out["lon"] < 0).any():
        out.loc[out["lon"] < 0, "lon"] = out.loc[out["lon"] < 0, "lon"] + 360
    return out


def compute_event_rankings(
    ph_points_df,
    buses_df,
    search_radius_km,
    decay_km,
    wind_norm_mode="global_max",
    name_map=None,
):
    if name_map is None:
        name_map = {}
    if ph_points_df.empty:
        raise ValueError("ph_points_df is empty.")
    if buses_df.empty:
        raise ValueError("buses_df is empty.")
    if search_radius_km <= 0 or decay_km <= 0:
        raise ValueError("search_radius_km and decay_km must be positive.")

    max_wind_global = ph_points_df["wind_kph"].max()
    if pd.isna(max_wind_global) or max_wind_global <= 0:
        max_wind_global = 1.0

    bus_lat = buses_df["lat"].to_numpy()
    bus_lon = buses_df["lon"].to_numpy()
    bus_id_arr = buses_df["bus_id"].to_numpy()
    bus_w_arr = buses_df["bus_weight"].to_numpy()

    bus_event_rows = []

    for storm_id, sgrp in ph_points_df.groupby("storm_id"):
        hazard_max = np.zeros(len(buses_df), dtype=float)

        if wind_norm_mode == "storm_max":
            storm_max = sgrp["wind_kph"].max()
            if pd.isna(storm_max) or storm_max <= 0:
                storm_max = 1.0
        else:
            storm_max = max_wind_global

        for _, sp in sgrp.iterrows():
            d = haversine_km(bus_lat, bus_lon, sp["lat"], sp["lon"])
            near = d <= search_radius_km
            if not np.any(near):
                continue

            wind_norm = float(sp["wind_kph"] / storm_max)
            hz = wind_norm * np.exp(-d[near] / decay_km)
            hazard_max[near] = np.maximum(hazard_max[near], hz)

        storm_year = sgrp["year"].min() if "year" in sgrp.columns else np.nan

        for i in range(len(buses_df)):
            bus_event_rows.append(
                {
                    "storm_id": storm_id,
                    "year": storm_year,
                    "bus_id": bus_id_arr[i],
                    "hazard_max": float(hazard_max[i]),
                    "bus_weight": float(bus_w_arr[i]),
                    "impact_bus": float(hazard_max[i] * bus_w_arr[i]),
                }
            )

    bus_event_df = pd.DataFrame(bus_event_rows)

    storm_rank = (
        bus_event_df.groupby("storm_id", as_index=False)
        .agg(
            impact_score=("impact_bus", "sum"),
            max_hazard=("hazard_max", "max"),
            mean_hazard=("hazard_max", "mean"),
            impacted_buses=("hazard_max", lambda x: int((x > 0).sum())),
            mean_bus_weight_impacted=("bus_weight", lambda x: float(np.mean(x)) if len(x) else np.nan),
        )
    )

    if "year" in bus_event_df.columns:
        year_map = bus_event_df.groupby("storm_id", as_index=False)["year"].min()
        storm_rank = storm_rank.merge(year_map, on="storm_id", how="left")

    storm_rank["storm_name"] = storm_rank["storm_id"].map(name_map).fillna("UNNAMED")
    storm_rank = storm_rank.sort_values("impact_score", ascending=False).reset_index(drop=True)
    storm_rank["rank"] = storm_rank.index + 1

    return storm_rank, bus_event_df


def precision_at_k(ranked_ids: list, positive_ids: set, k: int) -> float:
    if k <= 0 or not ranked_ids:
        return 0.0
    top = ranked_ids[:k]
    return sum(1 for sid in top if sid in positive_ids) / min(k, len(top))


def logistic_fail_prob(h, center=FAIL_CENTER, steepness=FAIL_STEEPNESS):
    h = np.asarray(h, dtype=float)
    return 1.0 / (1.0 + np.exp(-steepness * (h - center)))

## Load IBTrACS and screen to the Philippines box

In [ ]:
ib = tracks.TrackDataset(basin="west_pacific", source="ibtracs")
name_map = storm_name_map_from_ib(ib)
print(f"Loaded {len(ib.data)} storms from IBTrACS West Pacific archive.")

rows = []
for storm_id, storm in ib.data.items():
    year = storm.get("year", storm.get("season"))
    if year is None:
        continue
    try:
        year = int(year)
    except (TypeError, ValueError):
        continue
    if year < START_YEAR:
        continue

    lat_list = storm.get("lat", [])
    lon_list = storm.get("lon", [])
    vmax_list = storm.get("vmax", [])
    time_list = storm.get("time", [])
    name = storm.get("name", "UNKNOWN")

    n_points = min(len(lat_list), len(lon_list), len(vmax_list))
    for point_index in range(n_points):
        lat = lat_list[point_index]
        lon = lon_list[point_index]
        vmax_kt = vmax_list[point_index]
        time = time_list[point_index] if point_index < len(time_list) else None

        if lat is None or lon is None or vmax_kt is None:
            continue
        try:
            lat = float(lat)
            lon = float(lon)
            vmax_kt = float(vmax_kt)
        except (TypeError, ValueError):
            continue

        if lon < 0:
            lon += 360

        rows.append(
            {
                "storm_id": str(storm_id),
                "name": str(name),
                "point_index": point_index,
                "time": time,
                "year": year,
                "lat": lat,
                "lon": lon,
                "wind_kt": vmax_kt,
                "wind_kph": vmax_kt * 1.852,
            }
        )

all_points = pd.DataFrame(rows).sort_values(["year", "storm_id", "point_index"]).reset_index(drop=True)

ph_points = all_points[
    all_points["lat"].between(PH_BOUNDS["lat_min"], PH_BOUNDS["lat_max"])
    & all_points["lon"].between(PH_BOUNDS["lon_min"], PH_BOUNDS["lon_max"])
].copy()

print(f"Total track points extracted: {len(all_points):,}")
print(f"Unique storms retained: {all_points['storm_id'].nunique():,}")
print(f"Track points within PH screening box: {len(ph_points):,}")
print(f"Unique storms with at least one point in PH box: {ph_points['storm_id'].nunique():,}")
print(f"Years: {ph_points['year'].min()}–{ph_points['year'].max()}")

display(ph_points.head())

## Module 1 — Philippines land mask

In [ ]:
from shapely.geometry import Point
from shapely import prepared
from shapely.ops import unary_union
import cartopy.io.shapereader as shpreader

def load_philippines_land_union(resolution="10m"):
    shpfilename = shpreader.natural_earth(
        resolution=resolution, category="cultural", name="admin_0_countries"
    )
    reader = shpreader.Reader(shpfilename)
    geoms = []
    for rec in reader.records():
        if rec.attributes.get("ADM0_A3") == "PHL":
            geoms.append(rec.geometry)
    if not geoms:
        raise RuntimeError("Philippines (PHL) not found in Natural Earth admin_0_countries.")
    return unary_union(geoms)

def _boundary_lonlat(land_geom, max_points=5000):
    boundary = land_geom.boundary
    if boundary.geom_type == "LineString":
        coords = np.array(boundary.coords)
    elif boundary.geom_type == "MultiLineString":
        coords = np.vstack([np.array(g.coords) for g in boundary.geoms])
    else:
        coords = np.array(boundary.coords)
    if len(coords) > max_points:
        idx = np.linspace(0, len(coords) - 1, max_points, dtype=int)
        coords = coords[idx]
    return coords[:, 0], coords[:, 1]

def _min_dist_coast_km(lat, lon, coast_lon, coast_lat):
    d = haversine_km(coast_lat, coast_lon, lat, lon)
    return float(np.min(d))

PH_LAND_UNION = load_philippines_land_union()
_prep_land = prepared.prep(PH_LAND_UNION)
_coast_lon, _coast_lat = _boundary_lonlat(PH_LAND_UNION)

_on = np.zeros(len(ph_points), dtype=bool)
_dcoast = np.zeros(len(ph_points), dtype=float)
for i in range(len(ph_points)):
    lon = float(ph_points["lon"].iloc[i])
    lat = float(ph_points["lat"].iloc[i])
    _on[i] = _prep_land.contains(Point(lon, lat))
    _dcoast[i] = _min_dist_coast_km(lat, lon, _coast_lon, _coast_lat)

ph_points = ph_points.copy()
ph_points["on_land"] = _on
ph_points["dist_to_coast_km"] = _dcoast
ph_points_land = ph_points.loc[ph_points["on_land"]].copy()
ph_points_sea = ph_points.loc[~ph_points["on_land"]].copy()

print(f"PH box track points: {len(ph_points):,}")
print(f"On Philippines land: {int(ph_points['on_land'].sum()):,}")
print(f"Unique storms (box): {ph_points['storm_id'].nunique():,}")
print(f"Unique storms (land points only): {ph_points_land['storm_id'].nunique():,}")

In [ ]:
ph_storm_summary = (
    ph_points.groupby(["storm_id", "name", "year"], as_index=False)
    .agg(
        n_points_in_box=("point_index", "count"),
        max_wind_kph_in_box=("wind_kph", "max"),
        mean_wind_kph_in_box=("wind_kph", "mean"),
        min_lat_in_box=("lat", "min"),
        max_lat_in_box=("lat", "max"),
        min_lon_in_box=("lon", "min"),
        max_lon_in_box=("lon", "max"),
    )
    .sort_values(["year", "max_wind_kph_in_box"], ascending=[True, False])
    .reset_index(drop=True)
)

display(ph_storm_summary.head(20))

## Population grid

In [ ]:
print("Population file:", XYZ_PATH)
worldpop = load_worldpop_xyz(XYZ_PATH, verbose=True)

pop_ph = worldpop[
    worldpop["lat"].between(PH_BOUNDS["lat_min"], PH_BOUNDS["lat_max"])
    & worldpop["lon"].between(PH_BOUNDS["lon_min"], PH_BOUNDS["lon_max"])
    & (worldpop["pop_value"] > 0)
].copy()

print(f"Rows after cleaning: {len(worldpop):,}")
print(f"Population rows in PH box: {len(pop_ph):,}")
print(f"Share of global rows retained: {len(pop_ph) / len(worldpop):.2%}")
display(pop_ph.head())
display(pop_ph["pop_value"].describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]))

## Buses and canonical bus weights

In [ ]:
buses = pd.read_csv(BUSES_CSV)
buses_v2 = buses.rename(columns={"name": "bus_id", "x": "lon", "y": "lat"}).copy()

for c in ["lon", "lat", "v_nom"]:
    buses_v2[c] = pd.to_numeric(buses_v2[c], errors="coerce")

buses_v2 = buses_v2.dropna(subset=["bus_id", "lon", "lat", "v_nom"]).copy()

vmax = buses_v2["v_nom"].max()
buses_v2["v_norm"] = 0.0 if vmax <= 0 else buses_v2["v_nom"] / vmax
buses_v2["bus_weight_v"] = 1.0 + ALPHA_VNOM * buses_v2["v_norm"]

bus_pop_vals = []
for _, b in buses_v2.iterrows():
    d = haversine_km(
        pop_ph["lat"].to_numpy(),
        pop_ph["lon"].to_numpy(),
        b["lat"],
        b["lon"],
    )
    local_pop = pop_ph.loc[d <= BUS_POP_RADIUS_KM, "pop_value"].sum()
    bus_pop_vals.append(local_pop)

buses_v2["bus_pop_local"] = bus_pop_vals
pmax = buses_v2["bus_pop_local"].max()
buses_v2["bus_pop_norm"] = 0.0 if pmax <= 0 else buses_v2["bus_pop_local"] / pmax
buses_v2["bus_weight_pop"] = 1.0 + buses_v2["bus_pop_norm"]
buses_v2["bus_weight"] = buses_v2["bus_weight_v"] * buses_v2["bus_weight_pop"]

display(
    buses_v2[
        ["bus_id", "v_nom", "bus_pop_local", "bus_weight_v", "bus_weight_pop", "bus_weight"]
    ].sort_values("bus_weight", ascending=False).head(15)
)

## Light network adjustment of canonical bus weight

In [ ]:
lines_net = pd.read_csv(LINES_CSV)
gens = pd.read_csv(GENERATORS_CSV)

buses_v2["bus_weight_pre_network"] = buses_v2["bus_weight"].copy()

deg = pd.concat([lines_net["bus0"], lines_net["bus1"]]).value_counts()
deg_series = buses_v2["bus_id"].map(deg).fillna(0).astype(float)
redundancy = np.sqrt(2.0 / np.maximum(deg_series.to_numpy(), 1.0))

cap_by_bus = gens.groupby("bus")["p_nom"].sum()
cap = buses_v2["bus_id"].map(cap_by_bus).fillna(0.0).to_numpy()
cmax = float(np.max(cap)) if len(cap) else 0.0
gen_norm = cap / cmax if cmax > 0 else np.zeros_like(cap)

w_deg = 1.0 - ALPHA_NET_DEG + ALPHA_NET_DEG * redundancy
w_gen = 1.0 + ALPHA_NET_GEN * gen_norm

buses_v2["graph_degree"] = deg_series.to_numpy()
buses_v2["bus_weight"] = buses_v2["bus_weight_pre_network"].to_numpy() * w_deg * w_gen

print(
    "bus_weight: pre-network mean",
    f"{buses_v2['bus_weight_pre_network'].mean():.4f},",
    "after network mean",
    f"{buses_v2['bus_weight'].mean():.4f}",
)
display(
    buses_v2[["bus_id", "graph_degree", "bus_weight_pre_network", "bus_weight"]]
    .sort_values("bus_weight", ascending=False)
    .head(15)
)

## Canonical ranking: `impact_score`

In [ ]:
ph_points_v2 = ph_points.copy().sort_values(["storm_id", "point_index"]).copy()
ph_points_v2["t"] = ph_points_v2.groupby("storm_id").cumcount()
if "year" not in ph_points_v2.columns:
    ph_points_v2["year"] = np.nan

storm_rank_v2, bus_event_df = compute_event_rankings(
    ph_points_v2,
    buses_v2,
    SEARCH_RADIUS_KM,
    DECAY_KM,
    name_map=name_map,
)

print("Ranked storms:", len(storm_rank_v2))
display(
    storm_rank_v2[
        ["rank", "storm_id", "storm_name", "year", "impact_score", "max_hazard", "mean_hazard", "impacted_buses"]
    ].head(20)
)

## Land-only vs full-box comparison

In [ ]:
ph_points_v2_land = ph_points_land.copy().sort_values(["storm_id", "point_index"]).copy()
ph_points_v2_land["t"] = ph_points_v2_land.groupby("storm_id").cumcount()
if "year" not in ph_points_v2_land.columns:
    ph_points_v2_land["year"] = np.nan

storm_rank_land, _ = compute_event_rankings(
    ph_points_v2_land,
    buses_v2,
    SEARCH_RADIUS_KM,
    DECAY_KM,
    name_map=name_map,
)

rank_compare = storm_rank_v2[["storm_id", "rank", "impact_score"]].merge(
    storm_rank_land[["storm_id", "rank", "impact_score"]],
    on="storm_id",
    how="inner",
    suffixes=("_box", "_land"),
)

rho, _ = spearmanr(rank_compare["rank_box"], rank_compare["rank_land"])
print(f"Spearman rho (box vs land-only ranks): {rho:.3f}  (N={len(rank_compare)})")
display(rank_compare.sort_values("rank_box").head(20))

## Simple canonical visual diagnostics

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
plot_df = storm_rank_v2.head(15).sort_values("impact_score")
ax.barh(
    plot_df["storm_name"] + " (" + plot_df["year"].astype(int).astype(str) + ")",
    plot_df["impact_score"],
)
ax.set_title("Top 15 storms by impact_score")
ax.set_xlabel("impact_score")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(storm_rank_v2["impacted_buses"], storm_rank_v2["impact_score"])
ax.set_title("Impact score vs impacted buses")
ax.set_xlabel("Impacted buses")
ax.set_ylabel("impact_score")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(storm_rank_v2["max_hazard"], storm_rank_v2["impact_score"])
ax.set_title("Impact score vs maximum bus hazard")
ax.set_xlabel("Maximum bus hazard")
ax.set_ylabel("impact_score")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Robustness

In [ ]:
search_radius_values = [100.0, 150.0, 200.0]
decay_values = [20.0, 30.0, 50.0, 75.0]
param_grid = list(product(search_radius_values, decay_values))

all_rankings = []
for search_radius_km, decay_km in param_grid:
    rank_df, _ = compute_event_rankings(
        ph_points_df=ph_points_v2,
        buses_df=buses_v2,
        search_radius_km=search_radius_km,
        decay_km=decay_km,
        name_map=name_map,
    )
    rank_df = rank_df.copy()
    rank_df["search_radius_km"] = search_radius_km
    rank_df["decay_km"] = decay_km
    all_rankings.append(rank_df)

robustness_df = pd.concat(all_rankings, ignore_index=True)
display(
    robustness_df[
        ["search_radius_km", "decay_km", "rank", "storm_name", "year", "impact_score"]
    ].head(20)
)

In [ ]:
setting_labels = []
top10_sets = {}

for (search_radius_km, decay_km), grp in robustness_df.groupby(["search_radius_km", "decay_km"]):
    label = f"R{int(search_radius_km)}_D{int(decay_km)}"
    setting_labels.append(label)
    top10_sets[label] = set(grp.nsmallest(10, "rank")["storm_id"])

overlap_rows = []
for a in setting_labels:
    for b in setting_labels:
        inter = len(top10_sets[a].intersection(top10_sets[b]))
        overlap_rows.append({"setting_a": a, "setting_b": b, "top10_overlap": inter})

overlap_df = pd.DataFrame(overlap_rows)
overlap_pivot = overlap_df.pivot(index="setting_a", columns="setting_b", values="top10_overlap")
display(overlap_pivot)

In [ ]:
grouped_rankings = {
    f"R{int(sr)}_D{int(dk)}": grp[["storm_id", "rank"]].rename(columns={"rank": f"rank_R{int(sr)}_D{int(dk)}"})
    for (sr, dk), grp in robustness_df.groupby(["search_radius_km", "decay_km"])
}

labels = list(grouped_rankings.keys())
spearman_rows = []
for i, a in enumerate(labels):
    for b in labels[i:]:
        merged = grouped_rankings[a].merge(grouped_rankings[b], on="storm_id", how="inner")
        col_a = merged.columns[1]
        col_b = merged.columns[2]
        rho, _ = spearmanr(merged[col_a], merged[col_b])
        spearman_rows.append({"setting_a": a, "setting_b": b, "spearman_rho": rho})
        if a != b:
            spearman_rows.append({"setting_a": b, "setting_b": a, "spearman_rho": rho})

spearman_df = pd.DataFrame(spearman_rows)
spearman_pivot = spearman_df.pivot(index="setting_a", columns="setting_b", values="spearman_rho")
display(spearman_pivot)

rank_summary = (
    robustness_df.groupby(["storm_id", "storm_name", "year"], as_index=False)
    .agg(
        mean_rank=("rank", "mean"),
        best_rank=("rank", "min"),
        worst_rank=("rank", "max"),
        appearances=("rank", "count"),
        mean_score=("impact_score", "mean"),
    )
    .sort_values(["mean_rank", "best_rank"])
    .reset_index(drop=True)
)

display(rank_summary.head(25))

## Known-event check and placebo

In [ ]:
KNOWN_MAJOR_STORM_IDS = {
    "WP182006",  # Xangsane
    "WP242013",  # Haiyan
    "WP262012",  # Bopha
    "WP302021",  # Rai
    "WP192020",  # Goni
}

ranked = storm_rank_v2["storm_id"].tolist()
print("Precision@k:")
for k in (5, 10, 20):
    print(f"  k={k:2d}: {precision_at_k(ranked, KNOWN_MAJOR_STORM_IDS, k):.3f}")

ps = rank_summary.copy()
ps["known_major_flag"] = ps["storm_id"].isin(KNOWN_MAJOR_STORM_IDS)
display(ps[["storm_id", "storm_name", "year", "mean_rank", "best_rank", "mean_score", "known_major_flag"]].head(30))

rng = np.random.default_rng(RANDOM_SEED)
placebo_rhos = []
for _ in range(PLACEBO_N):
    df = storm_rank_v2.copy()
    scores = df["impact_score"].to_numpy().copy()
    rng.shuffle(scores)
    df["_s"] = scores
    df = df.sort_values("_s", ascending=False).reset_index(drop=True)
    df["rank_p"] = df.index + 1
    m = storm_rank_v2[["storm_id", "rank"]].merge(df[["storm_id", "rank_p"]], on="storm_id")
    placebo_rhos.append(spearmanr(m["rank"], m["rank_p"])[0])

print(f"Placebo mean Spearman over {PLACEBO_N} draws: {np.mean(placebo_rhos):.4f}")

## Land vs sea split

In [ ]:
ph_points_v2_sea = ph_points_sea.copy().sort_values(["storm_id", "point_index"]).copy()
ph_points_v2_sea["t"] = ph_points_v2_sea.groupby("storm_id").cumcount()
if "year" not in ph_points_v2_sea.columns:
    ph_points_v2_sea["year"] = np.nan

if len(ph_points_v2_sea) == 0:
    storm_rank_sea = pd.DataFrame(columns=["storm_id", "impact_score", "rank"])
else:
    storm_rank_sea, _ = compute_event_rankings(
        ph_points_v2_sea, buses_v2, SEARCH_RADIUS_KM, DECAY_KM, name_map=name_map
    )

geo_val = (
    storm_rank_v2[["storm_id", "storm_name", "year", "rank", "impact_score"]]
    .merge(
        storm_rank_land[["storm_id", "impact_score"]].rename(columns={"impact_score": "impact_land_only"}),
        on="storm_id",
        how="left",
    )
    .merge(
        storm_rank_sea[["storm_id", "impact_score"]].rename(columns={"impact_score": "impact_sea_only"}),
        on="storm_id",
        how="left",
    )
)

geo_val["impact_land_only"] = geo_val["impact_land_only"].fillna(0.0)
geo_val["impact_sea_only"] = geo_val["impact_sea_only"].fillna(0.0)
_tot = geo_val["impact_land_only"] + geo_val["impact_sea_only"]
geo_val["land_share_of_split"] = np.where(_tot > 0, geo_val["impact_land_only"] / _tot, np.nan)

display(geo_val.head(25))

## External validation benchmark table

In [ ]:
BENCHMARK_EXTERNAL_CSV = REPO_ROOT / "data/storm_impact_benchmark_external.csv"

sum_part = ph_storm_summary[
    ["storm_id", "n_points_in_box", "max_wind_kph_in_box", "mean_wind_kph_in_box"]
].copy()

validation_benchmark_table = storm_rank_v2.merge(sum_part, on="storm_id", how="left")

validation_benchmark_table = validation_benchmark_table.merge(
    geo_val[["storm_id", "land_share_of_split", "impact_land_only", "impact_sea_only"]],
    on="storm_id",
    how="left",
)

ext_cols = [
    "storm_id",
    "deaths",
    "affected_people",
    "damage_usd",
    "grid_outage_proxy",
    "restoration_proxy",
    "notes",
    "source",
]

if BENCHMARK_EXTERNAL_CSV.exists():
    ext_df = pd.read_csv(BENCHMARK_EXTERNAL_CSV)
    ext_df["storm_id"] = ext_df["storm_id"].astype(str)
    for c in ext_cols[1:]:
        if c not in ext_df.columns:
            ext_df[c] = np.nan
    ext_df = ext_df[ext_cols].drop_duplicates(subset=["storm_id"], keep="last")
else:
    ext_df = pd.DataFrame({"storm_id": validation_benchmark_table["storm_id"].astype(str)})
    for c in ext_cols[1:]:
        ext_df[c] = np.nan
    BENCHMARK_EXTERNAL_CSV.parent.mkdir(parents=True, exist_ok=True)
    ext_df.to_csv(BENCHMARK_EXTERNAL_CSV, index=False)
    print("Created empty template:", BENCHMARK_EXTERNAL_CSV)

validation_benchmark_table["storm_id"] = validation_benchmark_table["storm_id"].astype(str)
validation_benchmark_table = validation_benchmark_table.merge(ext_df, on="storm_id", how="left")

out_bench = OUTPUT_DIR / "storm_validation_benchmark.csv"
validation_benchmark_table.to_csv(out_bench, index=False)
print("Saved:", out_bench.resolve())

display(validation_benchmark_table.head(25))

## Optional joined external validation against curated master file

In [ ]:
MASTER_PATH = REPO_ROOT / "data/storm_validation_master.csv"

if MASTER_PATH.exists():
    val = pd.read_csv(MASTER_PATH).copy()
    val["storm_id"] = val["storm_id"].astype(str)

    wanted_val_cols = [
        "storm_id",
        "deaths",
        "affected_people",
        "damage_usd",
        "grid_outage_proxy",
        "restoration_proxy",
        "notes",
        "source",
    ]
    for c in wanted_val_cols:
        if c not in val.columns:
            val[c] = np.nan
    val = val[wanted_val_cols].drop_duplicates(subset=["storm_id"], keep="last")

    _sum = ph_storm_summary[
        ["storm_id", "max_wind_kph_in_box", "n_points_in_box", "mean_wind_kph_in_box"]
    ].copy()
    _sum["storm_id"] = _sum["storm_id"].astype(str)

    bench = storm_rank_v2.copy()
    bench["storm_id"] = bench["storm_id"].astype(str)
    bench = bench.merge(val, on="storm_id", how="left")
    bench = bench.merge(_sum, on="storm_id", how="left")

    for col in [
        "deaths", "affected_people", "damage_usd", "grid_outage_proxy", "restoration_proxy",
        "max_wind_kph_in_box", "n_points_in_box", "mean_wind_kph_in_box", "impact_score"
    ]:
        if col in bench.columns:
            bench[col] = pd.to_numeric(bench[col], errors="coerce")

    print("Spearman vs impact_score:")
    for col in ["deaths", "affected_people", "damage_usd", "grid_outage_proxy", "restoration_proxy"]:
        sub = bench.dropna(subset=[col, "impact_score"])
        if len(sub) > 2:
            rho, p = spearmanr(sub["impact_score"], sub[col])
            print(f"{col}: Spearman = {rho:.3f} (p={p:.3g}, n={len(sub)})")
        else:
            print(f"{col}: need >2 paired rows, got {len(sub)}")

    display(bench.head(20))

    out_validation = OUTPUT_DIR / "storm_validation_joined_debug.csv"
    bench.to_csv(out_validation, index=False)
    print("Saved merged validation table to:", out_validation.resolve())
else:
    print("No storm_validation_master.csv found. Skipping joined external validation.")

## Population-sensitive extension

In [ ]:
bus_pop_layer = buses_v2.copy()
bus_pop_layer["bus_pop_local"] = pd.to_numeric(bus_pop_layer["bus_pop_local"], errors="coerce").fillna(0.0)

pop_max = bus_pop_layer["bus_pop_local"].max()
bus_pop_layer["w_pop_linear"] = 1.0 if (pd.isna(pop_max) or pop_max <= 0) else 1.0 + (bus_pop_layer["bus_pop_local"] / pop_max)

storm_bus_pop = bus_event_df.copy().merge(
    bus_pop_layer[["bus_id", "bus_pop_local", "w_pop_linear"]],
    on="bus_id",
    how="left"
)

storm_bus_pop["bus_pop_local"] = pd.to_numeric(storm_bus_pop["bus_pop_local"], errors="coerce").fillna(0.0)
storm_bus_pop["w_pop_linear"] = pd.to_numeric(storm_bus_pop["w_pop_linear"], errors="coerce").fillna(1.0)

storm_bus_pop["impact_pop_linear"] = storm_bus_pop["hazard_max"] * storm_bus_pop["w_pop_linear"]

event_pop_index = (
    storm_bus_pop.groupby("storm_id", as_index=False)
    .agg(
        n_buses_hit=("hazard_max", lambda s: int((s > 0).sum())),
        hazard_sum=("hazard_max", "sum"),
        pop_impact_sum_linear=("impact_pop_linear", "sum"),
        max_bus_pop_local=("bus_pop_local", "max"),
    )
    .merge(
        storm_rank_v2[["storm_id", "storm_name", "year", "rank", "impact_score"]],
        on="storm_id",
        how="left",
    )
    .sort_values("pop_impact_sum_linear", ascending=False)
    .reset_index(drop=True)
)

event_pop_index["rank_pop_linear"] = event_pop_index.index + 1
display(event_pop_index.head(20))

## Network-aware extension

In [ ]:
import networkx as nx

graph_lines = lines_net[["bus0", "bus1"]].dropna().copy()
graph_lines["bus0"] = graph_lines["bus0"].astype(str)
graph_lines["bus1"] = graph_lines["bus1"].astype(str)
graph_lines = graph_lines[graph_lines["bus0"] != graph_lines["bus1"]].copy()

G = nx.Graph()
for bus_id in buses_v2["bus_id"].astype(str):
    G.add_node(bus_id)
for _, row in graph_lines.iterrows():
    G.add_edge(row["bus0"], row["bus1"])

deg_cent = nx.degree_centrality(G)
bet_cent = nx.betweenness_centrality(G, normalized=True)
close_cent = nx.closeness_centrality(G)

network_metrics = pd.DataFrame({
    "bus_id": list(G.nodes()),
    "degree_centrality": [deg_cent.get(n, 0.0) for n in G.nodes()],
    "betweenness_centrality": [bet_cent.get(n, 0.0) for n in G.nodes()],
    "closeness_centrality": [close_cent.get(n, 0.0) for n in G.nodes()],
})

gen_by_bus = (
    gens.groupby("bus", as_index=False)["p_nom"]
    .sum()
    .rename(columns={"bus": "bus_id", "p_nom": "gen_p_nom"})
)
gen_by_bus["bus_id"] = gen_by_bus["bus_id"].astype(str)

network_metrics = network_metrics.merge(gen_by_bus, on="bus_id", how="left")
network_metrics["gen_p_nom"] = pd.to_numeric(network_metrics["gen_p_nom"], errors="coerce").fillna(0.0)

def minmax01(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0.0)
    smin = s.min()
    smax = s.max()
    if pd.isna(smax) or smax <= smin:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - smin) / (smax - smin)

network_metrics["deg_norm"] = minmax01(network_metrics["degree_centrality"])
network_metrics["bet_norm"] = minmax01(network_metrics["betweenness_centrality"])
network_metrics["close_norm"] = minmax01(network_metrics["closeness_centrality"])
network_metrics["gen_norm"] = minmax01(network_metrics["gen_p_nom"])

network_metrics["network_criticality_score"] = (
    0.20 * network_metrics["deg_norm"]
    + 0.45 * network_metrics["bet_norm"]
    + 0.15 * network_metrics["close_norm"]
    + 0.20 * network_metrics["gen_norm"]
)
network_metrics["bus_weight_network_crit"] = 1.0 + network_metrics["network_criticality_score"]

storm_bus_network = storm_bus_pop.copy()
storm_bus_network["bus_id"] = storm_bus_network["bus_id"].astype(str)
storm_bus_network = storm_bus_network.merge(
    network_metrics[["bus_id", "network_criticality_score", "bus_weight_network_crit"]],
    on="bus_id",
    how="left",
)

storm_bus_network["bus_weight_network_crit"] = pd.to_numeric(
    storm_bus_network["bus_weight_network_crit"], errors="coerce"
).fillna(1.0)

storm_bus_network["impact_network_crit"] = (
    storm_bus_network["hazard_max"] * storm_bus_network["bus_weight_network_crit"]
)

event_network_index = (
    storm_bus_network.groupby("storm_id", as_index=False)
    .agg(
        hazard_sum=("hazard_max", "sum"),
        network_impact_sum=("impact_network_crit", "sum"),
        max_network_criticality_hit=("network_criticality_score", "max"),
        mean_network_criticality_hit=("network_criticality_score", lambda s: float(np.mean(s[s > 0])) if np.any(s > 0) else 0.0),
        n_buses_hit=("hazard_max", lambda s: int((s > 0).sum())),
    )
    .merge(
        storm_rank_v2[["storm_id", "storm_name", "year", "rank", "impact_score"]],
        on="storm_id",
        how="left",
    )
    .sort_values("network_impact_sum", ascending=False)
    .reset_index(drop=True)
)

event_network_index["rank_network"] = event_network_index.index + 1
display(event_network_index.head(20))

## Functional impact extension

In [ ]:
func_df = storm_bus_network.copy()

for col in ["bus_pop_local", "bus_weight", "bus_weight_pre_network"]:
    if col in func_df.columns:
        func_df[col] = pd.to_numeric(func_df[col], errors="coerce")
    else:
        func_df[col] = np.nan

func_df["bus_pop_local"] = func_df["bus_pop_local"].fillna(0.0)
func_df["bus_weight"] = func_df["bus_weight"].fillna(1.0)
func_df["bus_weight_pre_network"] = func_df["bus_weight_pre_network"].fillna(1.0)

if "bus_weight_network_crit" in func_df.columns:
    func_df["bus_weight_network_crit"] = pd.to_numeric(
        func_df["bus_weight_network_crit"], errors="coerce"
    ).fillna(1.0)
else:
    func_df["bus_weight_network_crit"] = 1.0

func_df["p_fail"] = logistic_fail_prob(func_df["hazard_max"])
func_df["expected_pop_affected"] = func_df["p_fail"] * func_df["bus_pop_local"]
func_df["expected_infra_loss"] = func_df["p_fail"] * func_df["bus_weight"]
func_df["expected_network_loss"] = func_df["p_fail"] * func_df["bus_weight_network_crit"]

max_pop_local = max(func_df["bus_pop_local"].max(), 1.0)
func_df["expected_composite_consequence"] = (
    func_df["expected_infra_loss"] * (1.0 + func_df["expected_pop_affected"] / max_pop_local)
)

event_functional_index = (
    func_df.groupby("storm_id", as_index=False)
    .agg(
        expected_pop_affected=("expected_pop_affected", "sum"),
        expected_infra_loss=("expected_infra_loss", "sum"),
        expected_network_loss=("expected_network_loss", "sum"),
        expected_composite_consequence=("expected_composite_consequence", "sum"),
        mean_p_fail=("p_fail", "mean"),
        max_p_fail=("p_fail", "max"),
        n_buses_with_pfail_20=("p_fail", lambda s: int((s >= 0.20).sum())),
        n_buses_with_pfail_50=("p_fail", lambda s: int((s >= 0.50).sum())),
        n_buses_with_pfail_80=("p_fail", lambda s: int((s >= 0.80).sum())),
    )
    .merge(
        storm_rank_v2[["storm_id", "storm_name", "year", "rank", "impact_score"]],
        on="storm_id",
        how="left",
    )
    .sort_values("expected_composite_consequence", ascending=False)
    .reset_index(drop=True)
)

event_functional_index["rank_functional"] = event_functional_index.index + 1
event_functional_index["rank_shift_functional"] = (
    event_functional_index["rank"] - event_functional_index["rank_functional"]
)

display(event_functional_index.head(25))

## Soft fragmentation proxy

In [ ]:
frag_soft_rows = []

for sid, grp in func_df.groupby("storm_id"):
    pf_map = grp.set_index("bus_id")["p_fail"].to_dict()

    survival = {
        str(n): 1.0 - float(pf_map.get(str(n), 0.0))
        for n in G.nodes()
    }

    components = list(nx.connected_components(G))
    comp_survival = []
    for comp in components:
        comp_survival.append(sum(survival[n] for n in comp))

    largest_component_survival = max(comp_survival) if comp_survival else 0.0
    total_survival = sum(survival.values())

    frac_remaining_soft = (
        largest_component_survival / total_survival if total_survival > 0 else 0.0
    )

    frag_soft_rows.append({
        "storm_id": sid,
        "fraction_remaining_soft": frac_remaining_soft,
        "total_survival_weight": total_survival,
        "largest_component_survival": largest_component_survival,
    })

fragment_soft_df = pd.DataFrame(frag_soft_rows).merge(
    event_functional_index[
        [
            "storm_id", "storm_name", "year", "rank", "rank_functional",
            "impact_score", "expected_composite_consequence",
            "expected_pop_affected", "expected_network_loss",
        ]
    ],
    on="storm_id",
    how="left"
)

display(fragment_soft_df.sort_values("fraction_remaining_soft").head(20))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    fragment_soft_df["expected_composite_consequence"],
    fragment_soft_df["fraction_remaining_soft"],
    alpha=0.6,
    s=35,
)

label_df = fragment_soft_df.nsmallest(12, "fraction_remaining_soft")
for _, r in label_df.iterrows():
    yr = int(r["year"]) if pd.notna(r["year"]) else -1
    ax.text(
        r["expected_composite_consequence"],
        r["fraction_remaining_soft"],
        f"{r['storm_name']} ({yr})",
        fontsize=8,
        alpha=0.9,
        ha="left",
        va="bottom",
    )

ax.set_xlabel("expected_composite_consequence")
ax.set_ylabel("fraction_remaining_soft")
ax.set_title("Functional consequence vs soft residual network integrity")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Canonical comparison table

In [ ]:
master_compare = (
    storm_rank_v2[
        ["storm_id", "storm_name", "year", "rank", "impact_score"]
    ]
    .merge(
        event_pop_index[["storm_id", "rank_pop_linear", "pop_impact_sum_linear"]],
        on="storm_id",
        how="left",
    )
    .merge(
        event_network_index[["storm_id", "rank_network", "network_impact_sum"]],
        on="storm_id",
        how="left",
    )
    .merge(
        event_functional_index[
            ["storm_id", "rank_functional", "expected_composite_consequence", "expected_pop_affected", "expected_network_loss"]
        ],
        on="storm_id",
        how="left",
    )
    .merge(
        fragment_soft_df[["storm_id", "fraction_remaining_soft"]],
        on="storm_id",
        how="left",
    )
)

master_compare["shift_pop"] = master_compare["rank"] - master_compare["rank_pop_linear"]
master_compare["shift_network"] = master_compare["rank"] - master_compare["rank_network"]
master_compare["shift_functional"] = master_compare["rank"] - master_compare["rank_functional"]

display(master_compare.sort_values("rank").head(25))

out_master = OUTPUT_DIR / "storm_master_compare.csv"
master_compare.to_csv(out_master, index=False)
print("Saved:", out_master.resolve())

## Suggested reading of the outputs

Use `master_compare` as the main interpretation table:

- **baseline rank** → grid-footprint screening
- **population rank** → storms that rise when exposed population matters more
- **network rank** → storms that rise when structurally important buses matter more
- **functional rank** → storms that rise when hazard is translated into failure probability and consequence proxy
- **fraction_remaining_soft** → storms that most reduce effective residual network integrity

This is the final bridge from the notebook to narrative interpretation.